## Datalab Semester 2, Sprint 3

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

# 1. Zoek de map op waar dit notebook-bestand staat
map_van_notebook = os.path.dirname(os.path.abspath('__file__'))

# 2. Maak het volledige pad naar de database
db_pad = os.path.join(map_van_notebook, 'database.sqlite')
conn = sqlite3.connect(db_pad)

## Opdracht 1

### 1A


1. Rubric-criterium: "Bij het beantwoorden van de vragen is gebruik gemaakt van de programmeervereisten" (Leerdoel: 1 Recht en ethiek)

    Oorspronkelijke beoordeling: Matig (M)
    
    Feedback van de docent: "Er worden nauwelijks functies gebruikt."
    
    Oorspronkelijke situatie: De code voor vraag 1A en 1B bestond uit losse, opeenvolgende codeblokken in de cellen. Er was geen programmastructuur 
    aanwezig en er stonden nog overbodige, verkennende prints in de cellen (zoals het printen van de complete tabellenlijst).
    Nieuwe situatie (Aangepast): De volledige code van Opdracht 1 is gerefactored. Alle subvragen zijn nu volledig herschreven in **zelfgedefinieerde functies** (`def`) die universeel herbruikbaar zijn. Elke functie is consequent voorzien van een professionele docstring (`Args` en `Returns`) zijn best practices. Overbodige prints zijn verwijderd voor een schone structuur.
    
    Locatie in het notebook: **Vraag 1A:** Functie `bereken_wedstrijden_per_seizoen` (Cel [8])
    Vraag 1B: Functie `bereken_wedstrijden_per_kalenderjaar` (Cel [10])
    Vraag 1C: Functie `bereken_ranglijst_alle_seizoenen` (Cel [12])
    Vraag 1D: Functie `toon_eindposities_team` (Cel [13])



---
2. Rubric-criterium: "Er is een juiste vergelijking met de competitie gemaakt" (Leerdoel: 7 Regressiemodel)
    
    Oorspronkelijke beoordeling: Onvoldoende (O)
    
    Feedback van de docent: "Er wordt alleen een ranglijst voor seizoen 2010/2011 gegeven."*
    
    Oorspronkelijke situatie: De SQL-query in de python-functie filterde de database direct op een specifiek jaar (`2010/2011`). Hierdoor berekende de functie de stand niet voor de rest van de competitie, wat in strijd was met de vraag om de punten *per seizoen* te tonen.
    Nieuwe situatie (Aangepast): De functie `bereken_ranglijst_alle_seizoenen` is zo herschreven dat deze de stand, totale punten en doelsaldo berekent voor **alle seizoenen** die in de database aanwezig zijn. De hardgecodeerde SQL filtering is verwijderd. Er wordt nu pas buiten en na de functie gefilterd wanneer er specifieke seizoensdata getoond moet worden, waardoor de complete competitiehistori blijft behouden.
    
    Locatie in het notebook: Cel [12] (onder het kopje `### 1C`)

---

3. Rubric-criterium: "Er is een juiste vergelijking met de competitie gemaakt" (Leerdoel: 7 Regressiemodel)

    Oorspronkelijke beoordeling: Onvoldoende (O) / Resultaat was 'Onvoledig' 
    
    Oorspronkelijke situatie: Onder vraag 1D ontbrak elke vorm van code. Er was geen SQL-query of Pandas-operatie aanwezig; er stond enkel een handgeschreven tekstblok waarin handmatig werd opgemerkt dat Barcelona op de eerste plaats was geëindigd.
    
    Nieuwe situatie (Aangepast): Er is een volledig nieuwe, geautomatiseerde functie geschreven genaamd `toon_eindposities_team`. Deze functie grijpt programmatisch terug op de complete competitiedata uit 1C, filtert de historie specifiek op de naam van ons team ('FC Barcelona') en berekent via een Pandas-groepering (`cumcount() + 1`) de exacte eindpositie per seizoen. De resultaten worden nu correct en dynamisch via een Pandas Dataframe gegenereerd.
    
    Locatie in het notebook: Cel [13] (onder het kopje `### 1D`)

---

4. Aanvullende Verbetering (Best Practices & Performance binnen 1C)

    Oorspronkelijke situatie: Er werd een trage en inefficiënte `.apply(axis=1)` loop gebruikt die wedstrijd voor wedstrijd (rij voor rij) door de database heen liep om de wedstrijdpunten te berekenen.
    
    Nieuwe situatie (Aangepast): Om de code beter te laten lopen, is de loop volledig verwijderd. De berekening van de wedstrijdpunten (3 voor winst, 1 voor gelijkspel, 0 voor verlies) is nu uitgevoerd middels **vectorisatie** met behulp van `np.select`. Dit maakt optimaal gebruik van de snelheid van de onderliggende C-libraries en verbetert de performance van het notebook.
    
    Locatie in het notebook: Cel [12] (binnen de functie `bereken_ranglijst_alle_seizoenen`)

In [18]:
def bereken_wedstrijden_per_seizoen(team_naam, connection):
    """
    Haalt het totaal aantal gespeelde wedstrijden (thuis en uit) op 
    per seizoen voor een specifiek team.
    
    Args:
        team_naam (str): De exacte lange naam van het team (bijv. 'FC Barcelona').
        connection (sqlite3.Connection): De actieve databaseverbinding.
        
    Returns:
        pd.DataFrame: Dataframe met de kolommen 'season' en 'aantal_wedstrijden'.
    """
    query = """
    SELECT 
        m.season, 
        COUNT(*) AS aantal_wedstrijden
    FROM Match m
    JOIN Team t_home ON m.home_team_api_id = t_home.team_api_id
    JOIN Team t_away ON m.away_team_api_id = t_away.team_api_id
    WHERE 
        t_home.team_long_name = :team
        OR t_away.team_long_name = :team
    GROUP BY 
        m.season;
    """
    
    return pd.read_sql_query(query, connection, params={"team": team_naam})

df_wedstrijden_barca = bereken_wedstrijden_per_seizoen('FC Barcelona', conn)
df_wedstrijden_barca

,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


### 1B

In [19]:
def bereken_wedstrijden_per_kalenderjaar(team_naam, kalenderjaar, connection):
    """
    Haalt het aantal gespeelde wedstrijden op voor een specifiek team 
    binnen een bepaald kalenderjaar, gegroepeerd per voetbalseizoen.
    
    Args:
        team_naam (str): De exacte lange naam van het team (bijv. 'FC Barcelona').
        kalenderjaar (str/int): Het kalenderjaar van de wedstrijden (bijv. 2010).
        connection (sqlite3.Connection): De actieve databaseverbinding.
        
    Returns:
        pd.DataFrame: Dataframe met de kolommen 'season' en 'aantal_wedstrijden'.
    """
    query = """
    SELECT 
        m.season,
        COUNT(*) AS aantal_wedstrijden
    FROM Match m
    JOIN Team t_home ON m.home_team_api_id = t_home.team_api_id
    JOIN Team t_away ON m.away_team_api_id = t_away.team_api_id
    WHERE 
        (t_home.team_long_name = :team OR t_away.team_long_name = :team)
        AND strftime('%Y', m.date) = :jaar
    GROUP BY 
        m.season
    ORDER BY 
        m.season;
    """
    
    params = {
        "team": team_naam, 
        "jaar": str(kalenderjaar)
    }
    
    return pd.read_sql_query(query, connection, params=params)

df_wedstrijden_2010 = bereken_wedstrijden_per_kalenderjaar('FC Barcelona', 2010, conn)
display(df_wedstrijden_2010)

,season,aantal_wedstrijden
0,2009/2010,23
1,2010/2011,16


### 1C


In [20]:
def bereken_ranglijst_alle_seizoenen(league_id, connection):
    """
    Genereert de eindranglijst (punten en doelsaldo) per team en per seizoen 
    voor een specifieke competitie.
    
    Args:
        league_id (int): De ID van de gekozen competitie.
        connection (sqlite3.Connection): De actieve databaseverbinding.
        
    Returns:
        pd.DataFrame: Dataframe met seizoenen, teamnamen, totale punten en doelsaldo,
                      gesorteerd op seizoen en positie.
    """
    query_matches = """
        SELECT season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal 
        FROM Match 
        WHERE league_id = :league
    """
    df_matches = pd.read_sql_query(query_matches, connection, params={"league": league_id})
    
    conditions = [
        df_matches['home_team_goal'] > df_matches['away_team_goal'],
        df_matches['home_team_goal'] < df_matches['away_team_goal']
    ]
    df_matches['home_points'] = np.select(conditions, [3, 0], default=1)
    df_matches['away_points'] = np.select(conditions, [0, 3], default=1)
    
    # 3. Splits en herstructureer naar een nette lijst per team-wedstrijd
    home_df = df_matches[['season', 'home_team_api_id', 'home_points', 'home_team_goal', 'away_team_goal']].rename(
        columns={'home_team_api_id': 'team_api_id', 'home_points': 'points', 'home_team_goal': 'voor', 'away_team_goal': 'tegen'}
    )
    away_df = df_matches[['season', 'away_team_api_id', 'away_points', 'away_team_goal', 'home_team_goal']].rename(
        columns={'away_team_api_id': 'team_api_id', 'away_points': 'points', 'away_team_goal': 'voor', 'home_team_goal': 'tegen'}
    )
    
    df_all_stats = pd.concat([home_df, away_df], ignore_index=True)
    
    ranglijst = df_all_stats.groupby(['season', 'team_api_id']).sum().reset_index()
    ranglijst['doelsaldo'] = ranglijst['voor'] - ranglijst['tegen']
    
    df_teams = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", connection)
    ranglijst = ranglijst.merge(df_teams, on='team_api_id')
    
    ranglijst = ranglijst.sort_values(by=['season', 'points', 'doelsaldo'], ascending=[True, False, False]).reset_index(drop=True)
    
    ranglijst['positie'] = ranglijst.groupby('season').cumcount() + 1
    
    return ranglijst[['season', 'positie', 'team_long_name', 'points', 'doelsaldo']]

LIGA_BBVA_ID = 21518 

df_ranglijsten = bereken_ranglijst_alle_seizoenen(LIGA_BBVA_ID, conn)

df_seizoen_2010 = df_ranglijsten[df_ranglijsten['season'] == '2010/2011'].reset_index(drop=True)
display(df_seizoen_2010)

,season,positie,team_long_name,points,doelsaldo
0,2010/2011,1,FC Barcelona,96,74
1,2010/2011,2,Real Madrid CF,92,69
2,2010/2011,3,Valencia CF,71,20
3,2010/2011,4,Villarreal CF,62,10
4,2010/2011,5,Atlético Madrid,58,9
5,2010/2011,6,Athletic Club de Bilbao,58,4
6,2010/2011,7,Sevilla FC,58,1
7,2010/2011,8,RCD Espanyol,49,-9
8,2010/2011,9,CA Osasuna,47,-1
9,2010/2011,10,Real Sporting de Gijón,47,-7


### 1D




In [21]:
def toon_eindposities_team(df_ranglijsten, team_naam):
    """
    Filtert de totale ranglijst om te tonen op welke positie een specifiek 
    team is geëindigd in elk seizoen.
    
    Args:
        df_ranglijsten (pd.DataFrame): De complete ranglijst van alle seizoenen (uit 1C).
        team_naam (str): De naam van het team (bijv. 'FC Barcelona').
        
    Returns:
        pd.DataFrame: Een overzicht van de posities en punten per seizoen.
    """
    team_historie = df_ranglijsten[df_ranglijsten['team_long_name'] == team_naam]
    
    team_historie = team_historie.sort_values(by='season').reset_index(drop=True)
    
    return team_historie[['season', 'positie', 'team_long_name', 'points']]

df_barca_posities = toon_eindposities_team(df_ranglijsten, 'FC Barcelona')
display(df_barca_posities)

,season,positie,team_long_name,points
0,2008/2009,1,FC Barcelona,87
1,2009/2010,1,FC Barcelona,99
2,2010/2011,1,FC Barcelona,96
3,2011/2012,2,FC Barcelona,91
4,2012/2013,1,FC Barcelona,100
5,2013/2014,2,FC Barcelona,87
6,2014/2015,1,FC Barcelona,94
7,2015/2016,1,FC Barcelona,91


----------------------------------------------------------------------------------------------


## Opdracht 2

### [HERKANSING AANPASSINGEN] - Opdracht 2 Verbeteringen

In het kader van de herkansing is Opdracht 2 vanaf hier volledig herzien en aangepast op basis van de feedback.
Elke aangepaste code-cel is hieronder voorzien van een toelichtende markdown-cel waarin de wijzigingen worden benoemd.

#### [Herkansing Aanpassing] - Herschreven conclusies en interpretaties gebaseerd op de 3 plots en gevonden data

In deze analyse hebben we onderzocht in hoeverre de verschillende tactische teameigenschappen (attributes) samenhangen met het aantal behaalde punten op de ranglijst. Hiervoor hebben we de data van 6 verschillende seizoenen gecombineerd (totaal 120 observaties), wat een betrouwbare analyse mogelijk maakt op basis van echte, gevonden data uit onze DataFrame.

Als we kijken naar de correlatiematrix en de regressie-analyse, vallen op basis van onze gevonden data de volgende punten op:

1. **Verdedigende Druk en Breedte (Defence Pressure & Team Width)**
   De teameigenschappen die het sterkst positief samenhangen met het aantal behaalde punten zijn `defenceTeamWidth` (r = 0.327) en `defencePressure` (r = 0.286). In de scatterplot voor verdedigende druk (Defence Pressure) zien we een duidelijke stijgende regressielijn: teams die bereid zijn om hoog op het veld druk te zetten en verdedigend de ruimtes breed te bezetten, behalen gemiddeld aanzienlijk meer punten op de ranglijst. Dit is logisch, want dominante teams spelen vaak met veel druk naar voren om de bal snel te heroveren. Wel zien we spreiding in de scatterplot, wat betekent dat druk zetten op zichzelf geen garantie is voor succes, maar het is wel een duidelijke trend onder de beter presterende teams.

2. **Opbouw via Passen (Build Up Play Passing)**
   Onze data laat een opvallende negatieve correlatie zien tussen het aantal punten en `buildUpPlayPassing` (r = -0.247). Omdat dit attribuut de gemiddelde pass-afstand in de opbouwfase meet, betekent de negatieve correlatie dat teams die met **kortere passes** opbouwen (lagere score) gemiddeld **meer punten** behalen. Korte passing (zoals het bekende positiespel) vereist veel technische kwaliteit en balbezit-dominantie. Teams die veel met de lange bal spelen (hogere pass-afstand) eindigen gemiddeld lager op de ranglijst.

3. **Tactiek versus Spelerskwaliteit**
   Hoewel we duidelijke trends zien (zoals bij druk zetten en pass-afstand), is de correlatie voor de meeste eigenschappen relatief matig (tussen de -0.25 en 0.33). Dit bevestigt dat de teamattributen in de database voornamelijk de *tactische speelstijl* (hoe een team speelt) meten, en niet direct de *individuele kwaliteit* van de spelers. Dit zie je ook terug bij Barcelona en Real Madrid: hoewel ze heel anders spelen (Barcelona met extreem korte passes van 34, Real Madrid met een directere stijl van 45), eindigen ze allebei steevast in de top van de competitie. De speelstijl is dus een tactische keuze, maar de kwaliteit van de uitvoering bepaalt uiteindelijk de definitieve plaats op de ranglijst.


*Aanpassing Herkansing:* De conclusies en interpretaties zijn volledig herschreven en gebaseerd op de 3 gemaakte plots en de daadwerkelijk gevonden correlaties uit de data.